# Sentence Transformer

In [1]:
from deep_translator import GoogleTranslator
from langdetect import detect

from sentence_transformers import SentenceTransformer
import torch
import os
from pathlib import Path
import faiss

import numpy as np
import pandas as pd
import pickle


PROJECT_ROOT = Path(f"{Path.cwd()}").resolve()

# Function to automatically translate a user's query to English
def translate_to_english(text):

    try:
        detected_lang = detect(text)
        if detected_lang != 'en':
            translated_text = GoogleTranslator(source='auto', target='en').translate(text)
            return translated_text
        return text
    
    except Exception as e:
        print(f"Error at trying to detect languaje: {e}")
        return text
    
# Paths to read and store processed data files
processed_dir = Path("..") / "data" / "processed"
file_path = processed_dir / "df_clean.parquet"
faiss_index_path = processed_dir / "games_embeddings_faiss_IP.index"
embeddings_file_path = processed_dir / "games_embeddings_sentrans.pkl"

df = pd.read_parquet(file_path)

# Convert list-based columns into readable text format
df["genres_text"] = df["genres"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
df["tags_text"] = df["tags"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")
df["categories_text"] = df["categories"].apply(lambda x: " ".join(x) if isinstance(x, list) else "")

# Create a single text column with all game information
df["info_compressed"] = ( df["name"].astype(str).fillna('') + " ; " + 
                          df["about_the_game"].astype(str).fillna('').str[:300] + " ; " +  # limit description to 300
                          df["genres_text"].fillna('') + " ; " +
                          df["tags_text"].fillna('') + df["tags_text"].fillna('') + " ; " +
                          df["categories_text"].fillna(''))

df["info_compressed"] = df["info_compressed"].str.lower()

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu" # Use GPU (CUDA with pytorch) if available, otherwise fall back to CPU

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" # If possible, use expandable segments for better memory management
torch.cuda.empty_cache() # Release unused cached GPU memory

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)

with torch.no_grad():  # Disable gradient computation
    embeddings_list = model.encode(df["info_compressed"].tolist(), batch_size=64, show_progress_bar=True)

# Convert embeddings to a NumPy array and normalize them for similarity search
embeddings_np = np.array(embeddings_list, dtype=np.float32)
faiss.normalize_L2(embeddings_np)

d = embeddings_np.shape[1]

# Creating the FAISS index with IndexFlatIP
index = faiss.IndexFlatIP(d)
index.add(embeddings_np)

# Save FAISS index to disk
try:
    faiss.write_index(index, faiss_index_path)
    print(f"FAISS stored in {faiss_index_path}")
except Exception as e:
    print(f"Error trying to store FAISS's index {e}")

# Save embeddings list to disk
try:
    with open(embeddings_file_path, "wb") as f:
        pickle.dump(embeddings_list, f)
    print(f"Embeddings stored in {embeddings_file_path}")
except Exception as e:
    print(f"Error trying to store embeddings {e}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\PabloR\Documents\trabajosGithub\4geeks\Final-Project-Videogame-Recommender\.venv\Lib\site-packages\torch\nn\modules\module.py:1367: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


Batches:   0%|          | 0/1916 [00:00<?, ?it/s]

FAISS stored in ..\data\processed\games_embeddings_faiss_IP.index
Embeddings stored in ..\data\processed\games_embeddings_sentrans.pkl


In [3]:
query = "juego de estrategia"
query_translated = translate_to_english(query)

#Generate embedding for the user's query
query_embedding = model.encode([query_translated], device=device, show_progress_bar=False)
query_embedding_np = np.array(query_embedding, dtype=np.float32)

# Retrieve the top 2000 similar games from FAISS index
k = 2000
distances, indices = index.search(query_embedding_np, k)

if indices.size == 0:
    print("No results found.")
else:
    results = []
    for rank, idx in enumerate(indices[0]):
        sim_score = distances[0][rank]
        results_df = df.iloc[idx].to_dict()
        results_df['similarity'] = sim_score # store similarity score for reference
        results.append(results_df)

    results_df = pd.DataFrame(results)
    display(results_df[['name', 'about_the_game', 'categories', 'genres', 'tags', 'developers', 'publishers', 'windows', 'linux', 'mac']].head().reset_index(drop=True))

,name,about_the_game,categories,genres,tags,developers,publishers,windows,linux,mac
0,World Strategy War,"Introduction: Welcome, enthusiasts of strategy...","[Multi-player, PvP, Online PvP, Camera Comfort...",[Strategy],"[strategy, action, arcade, tower defense, stra...",[Mete Deniz],[MPARSDENIZ],True,True,False
1,Regular Strategy Game,Regular Strategy Game is strategy puzzle game ...,"[Single-player, Family Sharing]","[Indie, Strategy]",[],[Dusa Softworks],[Dusa Softworks],True,False,True
2,High Strategy: Oradros,"The first High Strategy game was Urukon, which...","[Single-player, Steam Achievements, Family Sha...","[Indie, Strategy]","[strategy, grand strategy, turn-based strategy...",[Iron Boar Labs Ltd.],[Iron Boar Labs Ltd.],True,False,False
3,To Victory,This is a single player strategy management ga...,"[Single-player, Family Sharing]","[Casual, Indie, Simulation, Strategy]","[casual, simulation, strategy, 4x, rts, grand ...",[Old_Gamer],[Old_Gamer],True,False,False
4,Hide and Shoot,――Dead or Discover Aim for victory without bei...,"[Multi-player, PvP, Online PvP, Cross-Platform...","[Free To Play, Strategy]","[strategy, turn-based strategy, wargame, pvp, ...","[Ginga software Co.,Ltd,]","[Ginga software Co.,Ltd,]",True,False,False
